# Phase 4 — final evaluation tables (§14.2, §19.2)

Thin display layer. All logic lives in [`final_evaluation.py`](final_evaluation.py) so the
weekly refresh through W4 is one function call rather than a re-execute-all, and so the
builder can be unit-tested and diffed.

```bash
python cesare/final_evaluation.py     # rebuild every table
```

**Source of truth:** [`FX_Carry_Strategy_Project_Plan.md`](FX_Carry_Strategy_Project_Plan.md).
Base version 1.1.0 (`summary(min_obs=)` passthrough + tenor-indexed roll leg); both fixes are
exact no-ops at the committed baseline, so every reconciliation target is unchanged.

In [1]:
import sys; sys.path.insert(0, "..")
import pandas as pd
pd.set_option("display.width", 220)

import final_evaluation as fe
from strategy import run
from strategy.episodes import ERAS, STRESS, compare_windows, report_windows

base = run()
base

<StrategyResult 'ALL' | 27 ccy | 2007-05-01->2026-06-30 | Sharpe gross 0.628 net 0.466>

## 1. Acceptance — the base still reconciles

Both W1 fixes touch `strategy/`, so this is the first thing to check, every time.

In [2]:
stats = base.summary(benchmark=None)
print(f"ALL gross {float(stats.iloc[0]['sharpe']):.4f}  net {float(stats.iloc[1]['sharpe']):.4f}"
      f"   (committed 0.6284 / 0.4659)")
print(f"turnover  {base.turnover:.6f}   (committed 0.675470)")
print(f"cost drag {base.cost_drag:.9f}   (committed 0.018146611)")
print(f"config    {base.config.describe()}")

ALL gross 0.6284  net 0.4659   (committed 0.6284 / 0.4659)
turnover  0.675470   (committed 0.675470)
cost drag 0.018146611   (committed 0.018146611)
config    {'universe': 'ALL', 'tenor': '1M', 'signal': 'carry', 'n_buckets': None, 'weighting': 'inv_vol', 'max_leg_share': 0.4, 'vol_window': 60, 'rebal': 'ME', 'vol_target': 0.1, 'lev_cap': 4.0, 'extra_lag': 0, 'costs': True, 'cost_multiple': 1.0, 'exposure': False, 'weight_overlay': None}


## 2. The stress windows — did the book preserve capital?

The primary lens since the desk's 2026-07-29 mandate (guardrail §6.8). Four of these eight
windows are under 120 trading days and returned **nothing at all** before the F1 fix, including
both windows the desk named personally and the taper tantrum — the second-worst window in the
sample. Annualised columns stay blank there by design.

In [3]:
stress = report_windows(base, STRESS, which="both")
stress[stress.basis == "net"][
    ["window", "n_days", "cum_return", "max_drawdown", "sharpe", "worst_day", "cost_drag"]
]

,window,n_days,cum_return,max_drawdown,sharpe,worst_day,cost_drag
1,gfc_2008,217,-0.059131,-0.178190,-0.430121,-0.040251,0.021353
3,euro_2011,392,-0.050951,-0.189635,-0.223985,-0.049060,0.019869
5,taper_2013,109,-0.128556,-0.190567,NaN,-0.040318,0.019032
7,china_em_2015,196,-0.051885,-0.098686,-0.552482,-0.029102,0.007507
9,covid_2020,64,-0.196187,-0.240379,NaN,-0.038106,0.027892
11,rates_2022,216,0.255431,-0.065564,2.750614,-0.021880,0.020749
13,oil_2026,85,0.100749,-0.017826,NaN,-0.017702,0.023415
15,semis_2026,65,0.116804,-0.017826,NaN,-0.017702,0.018322


## 3. The eras — where did the P&L come from?

`ERAS` partitions the sample, so `share_pnl` sums to exactly 1.0. That is what makes
"this era produced X% of the book's return" an honest statement rather than a cherry-pick.

In [4]:
eras = report_windows(base, ERAS, which="both")
net = eras[eras.basis == "net"]
print(f"share_pnl sums to {net['share_pnl'].sum():.15f}")
net[["window", "n_days", "cum_return", "ann_return", "sharpe", "max_drawdown", "share_pnl"]]

share_pnl sums to 1.000000000000000


,window,n_days,cum_return,ann_return,sharpe,max_drawdown,share_pnl
1,pre-crisis 2007-08,349,0.239158,0.161141,1.438267,-0.084149,0.215663
3,GFC 2008-09,217,-0.059131,-0.060778,-0.430121,-0.178190,-0.050577
5,recovery 2009-11,522,0.063632,0.034884,0.345215,-0.092528,0.069829
7,euro crisis 2011-12,392,-0.050951,-0.026568,-0.223985,-0.189635,-0.039939
9,taper + EM 2013-16,1044,0.117083,0.033129,0.293140,-0.229717,0.132634
11,calm 2017-19,782,0.059190,0.023924,0.230465,-0.162071,0.071743
13,covid 2020,262,-0.163072,-0.162772,-1.255718,-0.268879,-0.163540
15,tightening 2021-23,781,0.247233,0.077342,0.704387,-0.181729,0.231638
17,recent 2024-26,652,0.709288,0.212995,1.992870,-0.110733,0.532549


## 4. Per-leg accrual — the Jul 15 desk ask

*"Decompose the rate differential, split short leg vs long leg."* Derived from `contrib`,
which is tested to sum to `gross` exactly; the acceptance criterion is reconciliation
**< 1e-12**, not proximity.

In [5]:
legs = fe.leg_table(base)
full = legs[legs.freq == "FULL"].iloc[0]
print(f"max|resid| over every frequency: {legs['resid'].abs().max():.2e}\n")
print("Annualised contribution, full sample:")
for k in ("carry_long", "carry_short", "spot_long", "spot_short", "total", "gross"):
    print(f"  {k:12s} {float(full[k]):+.4%}")

annual = legs[legs.freq == "YE"]
losing = annual[annual.gross < 0]
print(f"\nLosing years: {len(losing)} of {len(annual)}; "
      f"of those, {int((losing.carry_long > 0).sum())} still accrued positive carry on the long leg.")
print("=> every losing year is a SPOT event on the long leg, never a carry event.")

max|resid| over every frequency: 3.88e-17

Annualised contribution, full sample:
  carry_long   +14.3124%
  carry_short  +2.4747%
  spot_long    -10.4302%
  spot_short   +0.6722%
  total        +7.0290%
  gross        +7.0290%

Losing years: 7 of 20; of those, 7 still accrued positive carry on the long leg.
=> every losing year is a SPOT event on the long leg, never a carry event.


## 5. Re-verdict — Stages 3 and 6 on the desk's objective (§19.3)

No re-runs; this re-reads committed CSVs. The project verdicted everything on Sharpe. The
desk's stated objective is capital preservation through the tail. Rule, fixed before computing:
**accept if the net Sharpe cost is ≤ 0.02 and the rule buys ≥ 1.0pp of MaxDD or ≥ 5% relative
CVaR₉₉.**

In [6]:
rv = fe.reverdict()
rv[["book", "rule", "sharpe", "d_sharpe", "max_drawdown", "d_maxdd_pp",
    "d_cvar99_rel", "verdict_sharpe_old", "verdict_tail_new", "flipped"]]

,book,rule,sharpe,d_sharpe,max_drawdown,d_maxdd_pp,d_cvar99_rel,verdict_sharpe_old,verdict_tail_new,flipped
0,ALL,Vol targeting (vs static),0.465923,0.010739,-0.293185,-11.087201,0.492864,adopt as sizing standard — no alpha claim,n/a — not a tail rule,False
1,ALL,VIX threshold (Stage 3 rule),0.441236,-0.024687,-0.249321,4.386370,-0.073347,tail-insurance-only,REJECT,False
2,ALL,"IV/RR binary, book-level",0.368963,-0.096960,-0.313398,-2.021348,-0.048602,combined REJECT,REJECT,False
3,ALL,IV/RR linear ramp,0.450378,-0.015545,-0.305481,-1.229564,-0.034015,dominated — reject,REJECT,False
4,ALL,Per-currency RR (longs only),0.456678,-0.009245,-0.276233,1.695201,-0.069287,tail-insurance-only — preferred,ACCEPT,True
5,G10,VIX threshold (Stage 3 rule),0.051946,-0.067187,-0.348586,3.373863,-0.130433,tail-insurance-only,REJECT,False
6,G10,"IV/RR binary, book-level",0.080479,-0.038654,-0.309971,7.235335,-0.163902,G10 tail-insurance-only,REJECT,False
7,G10,IV/RR linear ramp,0.108518,-0.010615,-0.361989,2.033570,-0.094992,dominated — reject,ACCEPT,True
8,G10,Per-currency RR (longs only),0.116150,-0.002983,-0.336887,4.543744,-0.072757,tail-insurance-only — preferred,ACCEPT,True
9,ALL,Regime: Crisis -> 0.5,0.469745,0.003822,-0.301776,-0.859067,-0.036699,REJECT as replacement,REJECT,False


In [7]:
for _, r in rv[rv.flipped].iterrows():
    print(f"{r['book']:4s} {r['rule']:42s} "
          f"Sharpe {r['d_sharpe']:+.4f}  MaxDD {r['d_maxdd_pp']:+.2f}pp  -> {r['verdict_tail_new']}")
print("\nCareful (§19.3): Stage 3's VIX *threshold* rule and Dafu's VIX *percentile* gate are")
print("different rules with similar drawdowns and very different Sharpes. Never merge them.")

ALL  Per-currency RR (longs only)               Sharpe -0.0092  MaxDD +1.70pp  -> ACCEPT
G10  IV/RR linear ramp                          Sharpe -0.0106  MaxDD +2.03pp  -> ACCEPT
G10  Per-currency RR (longs only)               Sharpe -0.0030  MaxDD +4.54pp  -> ACCEPT
ALL  Regime: Moderate -> 0.5, Crisis -> 0.0     Sharpe +0.0171  MaxDD +3.75pp  -> ACCEPT
ALL  VIX percentile gate (Dafu, p80/756d)       Sharpe -0.0007  MaxDD +4.82pp  -> ACCEPT

Careful (§19.3): Stage 3's VIX *threshold* rule and Dafu's VIX *percentile* gate are
different rules with similar drawdowns and very different Sharpes. Never merge them.


## 6. D6 term structure, re-priced

`tenor_sweep.csv` was cited by §17.3 but never written. It exists now, and its **net** column
is meaningful for the first time: before the F2 fix the roll leg was billed on the rebalance
grid, so a 12M forward paid twelve rolls a year of the 12M points spread and the drag *rose*
to 4.84%/yr while turnover *fell* (guardrail §6.11).

In [8]:
fe.tenor_sweep()

,tenor,gross_sharpe,net_sharpe,ann_return_net,ann_vol_net,max_drawdown_net,turnover,cost_drag,window_start,window_end
0,1M,0.628442,0.465923,0.052143,0.111914,-0.293185,0.675470,0.018147,2007-05-01,2026-06-30
1,3M,0.487503,0.350129,0.039537,0.112921,-0.326814,0.525908,0.015476,2007-05-01,2026-06-30
2,6M,0.514790,0.369708,0.041988,0.113571,-0.314240,0.450131,0.016446,2007-05-01,2026-06-30
3,12M,0.565674,0.399494,0.044978,0.112587,-0.311932,0.425905,0.018663,2007-05-01,2026-06-30


## 7. `final_comparison.csv` — the living artifact

Every named variant across all six workstreams, assembled from committed CSVs. Rows that do
**not** reconcile to the shared base are kept and flagged (`on_base=False`), never dropped —
a teammate's book disagreeing with the base is the §18 finding, not noise to hide.

In [9]:
fc = fe.final_comparison()
print(fc.groupby(["owner", "on_base"]).size().to_string())
fc[(fc.basis == "net") & fc.on_base].nlargest(12, "sharpe")[
    ["owner", "workstream", "variant", "sharpe", "max_drawdown", "cost_drag"]
]

owner   on_base
Arjun   False        5
Cesare  True       134
Dafu    True        14
Oleg    False        1
Theo    False        1
Vidhi   False        3


,owner,workstream,variant,sharpe,max_drawdown,cost_drag
13,Cesare,Phase 3 D1 skew (null),U21_carry,0.496238,-0.257568,0.013806
127,Cesare,Stage 6 regimes,ALL_reg_mod,0.483048,-0.255693,0.017279
125,Cesare,Stage 6 regimes,ALL_reg_half,0.469745,-0.301776,0.018242
129,Cesare,Stage 6 regimes,ALL_reg_off,0.465972,-0.310398,0.018421
6,Cesare,Phase 3 D1 skew (null),ALL_carry,0.465923,-0.293185,0.018147
42,Cesare,Phase 3 D6 term structure (null),tenor_1M,0.465923,-0.293185,0.018147
58,Cesare,Stage 3 dynamic carry,ALL_voltgt,0.465923,-0.293185,0.018147
78,Cesare,Stage 4 portfolio construction,ALL_inv_vol,0.465923,-0.293185,0.018121
89,Cesare,Stage 5 momentum,ALL_carry,0.465923,-0.293185,0.018121
137,Cesare,Stage 6 regimes,ALL_voltgt,0.465923,-0.293185,0.018147


In [10]:
compare_windows({"baseline": base, "G10": run("G10")}, STRESS, metric="max_drawdown")

,baseline,G10
window,,
gfc_2008,-0.178190,-0.234520
euro_2011,-0.189635,-0.101905
taper_2013,-0.190567,-0.129318
china_em_2015,-0.098686,-0.119899
covid_2020,-0.240379,-0.229259
rates_2022,-0.065564,-0.104996
oil_2026,-0.017826,-0.022502
semis_2026,-0.017826,-0.033692
